# Agente Funcional ChileAtiende — EP2 ISY0101 (Google Gemini 3.8 Flash)
**Evaluación Parcial N°2 — Ingeniería de Soluciones con IA**

- **Modelo LLM:** Google Gemini 3.8 Flash (`gemini-3.8-flash`), modelo oficial activo para Google AI Studio.
- **Base Vectorial (RAG):** Embeddings locales de alta velocidad (`all-MiniLM-L6-v2`), 0% de errores de API o cuotas.

Ejecuta las celdas en orden del 1 al 6.

## Celda 1 — Instalar dependencias

In [ ]:
# Instalacion de dependencias requeridas
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters chromadb sentence-transformers
print('Dependencias instaladas exitosamente.')

## Celda 2 — Configurar API Key de Google AI Studio
Lee tu clave automáticamente desde **Secrets** (nombre: `GOOGLE_API_KEY`).

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

api_key = ''

# 1. Intenta leer desde Secrets de Google Colab
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    pass

# 2. Si no esta en Secrets, usa la variable
if not api_key or api_key == 'PEGA_TU_API_KEY_AQUI':
    api_key = 'PEGA_TU_API_KEY_AQUI'

if not api_key or api_key == 'PEGA_TU_API_KEY_AQUI' or len(api_key) < 10:
    print('ATENCION: Falta configurar GOOGLE_API_KEY en Secrets.')
else:
    os.environ['GOOGLE_API_KEY'] = api_key
    print(f'GOOGLE_API_KEY detectada correctamente (termina en ...{api_key[-4:]})')

# Modelo oficial activo de Google
ACTIVE_GEMINI_MODEL = 'gemini-3.8-flash'
print(f'Modelo Gemini activo para el agente: {ACTIVE_GEMINI_MODEL}')

## Celda 3 — Base de Conocimientos (Trámites ChileAtiende)

In [ ]:
import os
os.makedirs('data', exist_ok=True)

fichas = '''=== PGU (Pension Garantizada Universal) ===
Descripcion: Beneficio del Estado que reemplaza los beneficios del Pilar Solidario, pagado por el IPS.
Requisitos:
- Tener 65 anos o mas de edad.
- No integrar el 10 por ciento mas rico de la poblacion.
- Acreditar residencia en Chile por 20 anos continuos o discontinuos, o 4 de los ultimos 5 anos previos a la solicitud.
- Contar con una pension base menor a 1.158.355 pesos.
Monto: El monto maximo es de 214.296 pesos mensuales.
Fuente: https://www.chileatiende.gob.cl/fichas/65066-pension-garantizada-universal-pgu

=== Bono al Trabajo de la Mujer (BTM) ===
Descripcion: Beneficio en dinero para mejorar los ingresos de las mujeres trabajadoras.
Requisitos:
- Tener entre 25 y 59 anos de edad.
- Pertenecer al 40 por ciento mas vulnerable segun el Registro Social de Hogares (RSH).
- Mantener las cotizaciones al dia.
- No trabajar en una Institucion del Estado.
Monto: Depende de la renta y puede pagarse de forma anual o mensual.
Fuente: https://www.chileatiende.gob.cl/fichas/1285-bono-al-trabajo-de-la-mujer

=== Subsidio de Arriendo ===
Descripcion: Beneficio habitacional del MINVU para arrendar una vivienda.
Requisitos:
- Ser mayor de 18 anos.
- No ser propietario de una vivienda.
- Pertenecer al 40 por ciento mas vulnerable segun el RSH.
- No haber sido beneficiado previamente con un subsidio habitacional.
Monto: Varia segun zona geografica y puede llegar hasta 5 UF mensuales.
Fuente: https://www.chileatiende.gob.cl/fichas/11281-subsidio-de-arriendo

=== Bono de Invierno ===
Descripcion: Beneficio anual pagado entre mayo y agosto para gastos de calefaccion.
Requisitos:
- Tener 65 anos o mas al 31 de mayo del ano del pago.
- Ser beneficiario de PGU, Pension Basica Solidaria o Aporte Previsional Solidario.
Monto: Varia cada ano, historicamente entre 50.000 y 100.000 pesos en un solo pago.
Fuente: https://www.chileatiende.gob.cl/fichas/2213-bono-de-invierno

=== Asignacion Familiar ===
Descripcion: Beneficio economico mensual para trabajadores y pensionados por cada carga familiar.
Requisitos:
- Ser trabajador dependiente, pensionado o imponente de alguna Caja de Prevision.
- Acreditar cargas familiares (hijos hasta 18 anos, o hasta 24 si son estudiantes).
Monto: Depende del tramo de renta del causante.
Fuente: https://www.chileatiende.gob.cl/fichas/14901-asignacion-familiar'''

with open('data/fichas_tramites.txt', 'w', encoding='utf-8') as f:
    f.write(fichas)

print('Base de conocimientos creada exitosamente.')

## Celda 4 — Vectorstore RAG (Embeddings Locales) y Herramientas del Agente

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.tools import tool

# 1. Embeddings Locales: 100% libre de cuotas y 404
class LocalEmbeddings:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

print("Cargando modelo de embeddings semanticos locales...")
embeddings = LocalEmbeddings()

# 2. Cargar y particionar fichas
loader = TextLoader("data/fichas_tramites.txt", encoding="utf-8")
docs = loader.load()
splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=80, separator="\n\n")
splits = splitter.split_documents(docs)

# 3. Construir VectorStore
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
print(f"VectorStore creado correctamente con {len(splits)} fragmentos.")

# 4. Definicion de las 3 Herramientas de la Evaluacion (IE1, IE2)
@tool
def buscar_tramite(consulta: str) -> str:
    """Busca informacion oficial sobre bonos, beneficios y tramites del Estado de Chile
    en la base de conocimientos de ChileAtiende. Usala cuando el ciudadano pregunte
    sobre requisitos, montos o postulaciones."""
    resultados = vectorstore.similarity_search(consulta, k=2)
    if not resultados:
        return "No se encontro informacion relevante en la base de datos."
    return "\n\n---\n\n".join([doc.page_content for doc in resultados])

@tool
def redactar_resumen(informacion: str) -> str:
    """Herramienta de redaccion y formateo: organiza y estructura los datos del
    tramite de forma accesible antes de la respuesta final."""
    lineas = [l.strip() for l in informacion.split("\n") if l.strip() and not l.startswith("Fuente:")]
    resumen_estructurado = "\n".join(f"- {linea}" for linea in lineas[:8])
    return f"Datos oficiales procesados para respuesta ciudadana:\n{resumen_estructurado}"

@tool
def derivar_ejecutivo(motivo: str) -> str:
    """Deriva al ciudadano a atencion humana cuando la consulta involucra datos
    personales (RUT) o supera el alcance del asistente."""
    return (
        "Entiendo que su consulta requiere atencion personalizada (" + motivo + "). "
        "Le recomendamos contactar a un ejecutivo de ChileAtiende:\n"
        "Call Center: 101 (lunes a viernes, 08:00 a 22:00 hrs)\n"
        "Web: https://www.chileatiende.gob.cl\n"
        "Un ejecutivo podra revisar su caso confidencialmente."
    )

TOOLS = [buscar_tramite, redactar_resumen, derivar_ejecutivo]
print("Herramientas listas:", [t.name for t in TOOLS])

## Celda 5 — Construir el Agente con Memoria de Corto Plazo

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from langchain_google_genai import ChatGoogleGenerativeAI

try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

try:
    from langchain.memory import ConversationBufferWindowMemory
except ImportError:
    from langchain_classic.memory import ConversationBufferWindowMemory

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Asegurar que el modelo siempre este definido
MODELO = globals().get("ACTIVE_GEMINI_MODEL", "gemini-3.8-flash")

SYSTEM_PROMPT = (
    "Eres un agente inteligente oficial de ChileAtiende, el sistema de atencion ciudadana del Estado de Chile.\n\n"
    "Tus responsabilidades:\n"
    "1. Responder consultas sobre bonos y beneficios usando la herramienta buscar_tramite.\n"
    "2. Dar formato accesible a la respuesta usando redactar_resumen.\n"
    "3. Derivar a atencion humana con derivar_ejecutivo si la consulta pide datos personales (como RUT) o esta fuera de la base.\n\n"
    "Reglas estrictas:\n"
    "- NUNCA inventes requisitos ni montos.\n"
    "- Usa un tono formal, claro y empatico.\n"
    "- Respeta el contexto previo de la conversacion.\n\n"
    "Historial previo: {chat_history}"
)

# Motor LLM de Google Gemini 3.8 Flash
llm = ChatGoogleGenerativeAI(model=MODELO, temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm=llm, tools=TOOLS, prompt=prompt)
memory = ConversationBufferWindowMemory(k=5, memory_key="chat_history", return_messages=True, input_key="input")

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

print(f"Agente ChileAtiende ({MODELO}) construido y listo para usar.")

## Celda 6 — Demostración de las 3 Consultas de Evaluación (IE7)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print('='*60)
print('  CONSULTA 1: Requisitos de un beneficio')
print('='*60)
r1 = agent_executor.invoke(
    {'input': 'Hola, tengo 67 anos. Cuales son los requisitos para la PGU?'}
)
print('\nRESPUESTA FINAL:')
print(r1['output'])

print('\n' + '='*60)
print('  CONSULTA 2: Prueba de memoria (continuidad sin repetir contexto)')
print('='*60)
r2 = agent_executor.invoke(
    {'input': 'Y cuanto dinero me darian cada mes?'}
)
print('\nRESPUESTA FINAL (memoria):')
print(r2['output'])

print('\n' + '='*60)
print('  CONSULTA 3: Derivacion por datos sensibles')
print('='*60)
r3 = agent_executor.invoke(
    {'input': 'Necesito saber el estado de mi solicitud, mi RUT es 12.345.678-9'}
)
print('\nRESPUESTA FINAL (derivacion):')
print(r3['output'])

## Celda 7 — Chat interactivo libre

In [ ]:
# ============================================================
# ESCRIBE TU PREGUNTA AQUÍ DENTRO DE LAS COMILLAS:
# ============================================================
mi_pregunta = "¿Cuánto dinero entrega el Bono de Invierno?"

print(f"Ciudadano: {mi_pregunta}\n")
print("Procesando consulta...")

respuesta = agent_executor.invoke({"input": mi_pregunta})

print("\n" + "="*60)
print("RESPUESTA FINAL DEL AGENTE CHILEATIENDE:")
print("="*60)
print(respuesta["output"])